# ДЗ 1. Рекомендательные системы

Смирнов Евгений Максимович, Б05-326.

Датасет: Amazon Review Data 2018, категория Musical Instruments, 5-core. Источник: https://cseweb.ucsd.edu/~jmcauley/datasets/amazon_v2/ , статья Ni, Li, McAuley, EMNLP 2019.

Запуск: скачать два файла в папку data рядом с ноутбуком, установить зависимости из requirements.txt и выполнить ячейки по порядку. Если файлов нет, следующая ячейка скачает их сама.

In [ ]:
import os, time, random, urllib.request
import numpy as np
import pandas as pd
import scipy.sparse as sp
import matplotlib.pyplot as plt
import torch

from common import read_jsonl_gz, parse_price, clean_categories, TrainSet, build_ground_truth, evaluate
from models import Popular, ItemKNN, IALS, EASE, FM, TimeSVDpp

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.set_num_threads(os.cpu_count())

DATA_DIR = "data"
BASE = "https://mcauleylab.ucsd.edu/public_datasets/data/amazon_v2/"
FILES = {
    "Musical_Instruments_5.json.gz": BASE + "categoryFilesSmall/Musical_Instruments_5.json.gz",
    "meta_Musical_Instruments.json.gz": BASE + "metaFiles2/meta_Musical_Instruments.json.gz",
}
os.makedirs(DATA_DIR, exist_ok=True)
for name, url in FILES.items():
    path = os.path.join(DATA_DIR, name)
    if not os.path.exists(path):
        urllib.request.urlretrieve(url, path)
K = 10

## 1. Данные

Отзыв на Amazon означает, что пользователь купил товар. Рекомендуем товары, которые пользователь купит в будущем. Релевантный объект: товар, купленный пользователем в оцениваемом периоде и оценённый на 4 или 5.

Поля отзывов: reviewerID это пользователь, asin это товар, overall это оценка 1–5, unixReviewTime это дата отзыва. Поля метаданных: asin, title, brand, category это путь в дереве категорий, price.

In [ ]:
reviews = read_jsonl_gz(os.path.join(DATA_DIR, "Musical_Instruments_5.json.gz"),
                        ["reviewerID", "asin", "overall", "unixReviewTime", "verified"])
meta = read_jsonl_gz(os.path.join(DATA_DIR, "meta_Musical_Instruments.json.gz"),
                     ["asin", "title", "brand", "category", "price"])
print("отзывов:", len(reviews), "товаров в метаданных:", len(meta))
reviews.head()

## 2. EDA и подготовка

In [ ]:
print("пропуски в отзывах:"); print(reviews.isna().sum())
df = reviews.rename(columns={"reviewerID": "user_id", "asin": "item_id", "overall": "rating", "unixReviewTime": "ts"})
df = df.dropna(subset=["user_id", "item_id", "rating", "ts"])
print("полные дубликаты строк:", df.duplicated().sum())
print("повторные пары пользователь-товар:", df.duplicated(["user_id", "item_id"]).sum())
df = df.sort_values("ts").drop_duplicates(["user_id", "item_id"], keep="first").reset_index(drop=True)
df["rating"] = df["rating"].astype(float)
df["day"] = (df["ts"] // 86400).astype(int)
df["date"] = pd.to_datetime(df["ts"], unit="s")
nu, ni = df.user_id.nunique(), df.item_id.nunique()
print(f"после очистки: {len(df)} взаимодействий, {nu} пользователей, {ni} товаров, плотность {len(df)/nu/ni:.5f}")
print("период:", df.date.min().date(), "-", df.date.max().date())

In [ ]:
fig, ax = plt.subplots(1, 4, figsize=(18, 3.5))
df.rating.value_counts().sort_index().plot.bar(ax=ax[0], title="оценки")
ax[1].hist(df.groupby("user_id").size(), bins=np.logspace(0, 3, 30)); ax[1].set_xscale("log"); ax[1].set_yscale("log"); ax[1].set_title("отзывов на пользователя")
ax[2].hist(df.groupby("item_id").size(), bins=np.logspace(0, 3.5, 30)); ax[2].set_xscale("log"); ax[2].set_yscale("log"); ax[2].set_title("отзывов на товар")
df.set_index("date").resample("QE").size().plot(ax=ax[3], title="отзывов в квартал")
plt.tight_layout(); plt.show()
print(df.groupby("user_id").size().describe().round(1).to_dict())
print(df.groupby("item_id").size().describe().round(1).to_dict())

In [ ]:
meta = meta.drop_duplicates("asin").set_index("asin")
meta = meta.loc[meta.index.intersection(df.item_id.unique())].copy()
meta["price_num"] = meta["price"].map(parse_price)
meta["cats"] = meta["category"].map(clean_categories)
meta["brand"] = meta["brand"].fillna("").astype(str).str.strip()
print("товаров с метаданными:", len(meta), "из", ni)
print("есть бренд:", round((meta.brand != "").mean(), 3), " есть цена:", round(meta.price_num.notna().mean(), 3),
      " есть категории:", round((meta.cats.str.len() > 0).mean(), 3))
print("брендов:", meta.brand.nunique())
print(meta.cats.map(lambda c: c[0] if c else "нет").value_counts().head(8))
meta.price_num.dropna().clip(upper=1000).plot.hist(bins=50, log=True, title="цена, $"); plt.show()

Выборка уже 5-core, объём попадает в требуемые рамки, поэтому подвыборку не делаем. Удаляем повторные отзывы на тот же товар, оставляя первый. Из метаданных берём только бренд, категории и цену. Поля rank, also_buy, also_view собраны в 2018 году и содержат информацию из будущего, их не используем.

## 3. Протокол оценки

Глобальное разбиение по времени: train до 80-го перцентиля дат, validation с 80-го до 90-го, test после 90-го. Гиперпараметры подбираем: обучение на train, оценка на validation. Финал: обучение на train+validation с выбранными параметрами, оценка на test.

Кандидаты: все товары из обучающей части. Уже купленные пользователем товары исключаем из выдачи. Новые пользователи без истории не оцениваются, им в продакшене отдали бы популярное. Новые товары не попадают в кандидаты, их покупки убираем из ground truth. Правила одинаковы для всех моделей, релевантность не зависит от таргета модели.

Метрики при K = 10. Основная NDCG@10, выбрана до подбора: учитывает позицию релевантного товара в выдаче. Также Recall@10, доля будущих покупок, попавших в top-10, и HR@10, доля пользователей хотя бы с одним попаданием. Coverage@10 показывает разнообразие выдачи.

In [ ]:
q80, q90 = df.ts.quantile([0.8, 0.9]).to_numpy()
train = df[df.ts < q80]; val = df[(df.ts >= q80) & (df.ts < q90)]; test = df[df.ts >= q90]
trainval = df[df.ts < q90]
print("границы:", pd.to_datetime(q80, unit="s").date(), pd.to_datetime(q90, unit="s").date())
print("размеры:", len(train), len(val), len(test))


def item_features(ts, meta, min_brand=5, n_price_bins=8):
    m = meta.reindex(ts.items)
    feats = []
    brand = m["brand"].fillna("")
    vc = brand.value_counts()
    brand = brand.where(brand.map(vc).fillna(0) >= min_brand, "other").replace("", "unknown")
    feats.append([["brand=" + b] for b in brand])
    cats = m["cats"].map(lambda c: c if isinstance(c, list) else [])
    feats.append([[f"cat{l}=" + x for l, x in enumerate(c)] for c in cats])
    p = np.log1p(m["price_num"].to_numpy(dtype=float))
    edges = np.nanquantile(p, np.linspace(0, 1, n_price_bins + 1)[1:-1]) if np.isfinite(p).any() else []
    pb = np.where(np.isnan(p), -1, np.searchsorted(edges, np.nan_to_num(p)))
    feats.append([[f"price={b}"] for b in pb])
    rows = [sum(parts, []) for parts in zip(*feats)]
    vocab = {f: j for j, f in enumerate(sorted({f for r in rows for f in r}))}
    data, ri, ci = [], [], []
    for r, fs in enumerate(rows):
        w = 1.0 / np.sqrt(len(fs))
        for f in fs:
            ri.append(r); ci.append(vocab[f]); data.append(w)
    return sp.csr_matrix((data, (ri, ci)), shape=(len(rows), len(vocab))), vocab


def user_features(ts, item_cat, n_bins=6):
    cnt = np.bincount(ts.u, minlength=ts.n_users)
    act = np.minimum(np.log2(cnt).astype(int), n_bins - 1)
    A = sp.csr_matrix((np.ones(ts.n_users), (np.arange(ts.n_users), act)), shape=(ts.n_users, n_bins))
    prof = ts.X @ item_cat
    prof = sp.diags(1.0 / np.maximum(np.asarray(prof.sum(1)).ravel(), 1e-9)) @ prof
    return sp.hstack([A, prof]).tocsr().astype(np.float32)


def make_trainset(part, t_rec):
    ts = TrainSet(part, t_rec=t_rec // 86400)
    ts.item_features, vocab = item_features(ts, meta)
    cat_cols = [j for f, j in vocab.items() if f.startswith("cat")]
    item_cat = (ts.item_features[:, cat_cols] > 0).astype(np.float32)
    ts.user_features = user_features(ts, item_cat)
    return ts


ts_val = make_trainset(train, q80)
ts_test = make_trainset(trainval, q90)
gt_val = build_ground_truth(ts_val, val)
gt_test = build_ground_truth(ts_test, test)
print("val: пользователей в оценке", len(gt_val), " test:", len(gt_test))
print("признаков товара:", ts_val.item_features.shape[1], " признаков пользователя:", ts_val.user_features.shape[1])

## 4. Модели

Popular. Бейзлайн для сравнения: самые популярные товары в train.

ItemKNN. Сходство товаров по бинарной матрице $X$: $s_{ij} = \frac{x_i^\top x_j}{\|x_i\|\|x_j\| + h}$, $h$ это сжатие. Для каждого товара оставляем $k$ ближайших соседей. Скор: $\hat r_{ui} = \sum_{j \in I_u} s_{ji}$.

iALS. $\min_{X,Y} \sum_{u,i} c_{ui}(p_{ui} - x_u^\top y_i)^2 + \lambda(\|X\|^2 + \|Y\|^2)$, где $p_{ui} = 1$ для покупок и 0 иначе, $c_{ui} = 1 + \alpha p_{ui}$. Попеременно решаем $x_u = (Y^\top C_u Y + \lambda I)^{-1} Y^\top C_u p_u$ и аналогично для $y_i$, используя $Y^\top C_u Y = Y^\top Y + Y^\top (C_u - I) Y$.

EASE. $\min_B \|X - XB\|_F^2 + \lambda\|B\|_F^2$ при $\mathrm{diag}(B) = 0$. Решение: $P = (X^\top X + \lambda I)^{-1}$, $B_{ij} = -P_{ij}/P_{jj}$. Скор: $XB$.

FM. $\hat y(x) = w_0 + \sum_f w_f x_f + \sum_{f<g} \langle v_f, v_g\rangle x_f x_g$. Признаки пользователя: ID, корзина активности в train, профиль категорий его прошлых покупок. Признаки товара: ID, бренд, категории всех уровней, корзина цены. Все признаки известны в момент рекомендации: профиль считается только по train, бренд, категории и цена это статичные свойства товара. Обучение BPR: $-\log\sigma(\hat y_{ui} - \hat y_{uj})$ со случайным негативом $j$.

timeSVD++. $\hat r_{ui}(t) = \mu + b_u + \alpha_u \mathrm{dev}_u(t) + b_{u,t} + (b_i + b_{i,\mathrm{Bin}(t)})(c_u + c_{u,t}) + q_i^\top\big(p_u + \alpha^p_u \mathrm{dev}_u(t) + |N(u)|^{-1/2}\sum_{j \in N(u)} y_j\big)$, $\mathrm{dev}_u(t) = \mathrm{sign}(t - t_u)|t - t_u|^{0.4}$. Время в днях, 30 бинов. Обучение MSE с L2 по оценкам train. Для рекомендаций считаем прогноз в момент начала оцениваемого периода, дневные компоненты при этом равны нулю, и ранжируем товары по прогнозу оценки.

## 5. Подбор гиперпараметров на validation

Основная метрика NDCG@10. Сетки небольшие, чтобы ноутбук считался за разумное время.

In [ ]:
GRIDS = {
    "ItemKNN": (ItemKNN, [dict(k=k, shrink=h) for k in [20, 100, 300] for h in [0, 20]]),
    "iALS": (IALS, [dict(factors=f, reg=r, alpha=a, iters=8) for f in [32, 64] for r in [1.0, 10.0] for a in [5.0, 20.0]]),
    "EASE": (EASE, [dict(reg=r) for r in [100.0, 300.0, 1000.0]]),
    "FM": (FM, [dict(dim=64, lr=2e-3, reg=r, epochs=e) for r in [1e-5, 1e-4] for e in [10, 20]]),
    "timeSVD++": (TimeSVDpp, [dict(factors=f, reg=r, epochs=150, lr=0.02) for f in [16, 64] for r in [1.0, 10.0]]),
}

log, best = [], {}
for name, (cls, grid) in GRIDS.items():
    for params in grid:
        t0 = time.time()
        m = cls(**params).fit(ts_val)
        res = evaluate(m, ts_val, gt_val, k=K)
        log.append({"model": name, **res, "params": params, "sec": round(time.time() - t0, 1)})
        print(name, params, round(res["NDCG@10"], 4), f"{time.time() - t0:.0f}s")
    rows = [r for r in log if r["model"] == name]
    best[name] = max(rows, key=lambda r: r["NDCG@10"])["params"]
pd.DataFrame(log).round(4)

## 6. Финальное сравнение на test

Обучаем на train+validation с лучшими параметрами.

In [ ]:
final, fitted = [], {}
for name, params in [("Popular", {})] + list(best.items()):
    cls = Popular if name == "Popular" else GRIDS[name][0]
    m = cls(**params).fit(ts_test)
    fitted[name] = m
    final.append({"model": name, **evaluate(m, ts_test, gt_test, k=K), "params": params})
results = pd.DataFrame(final).set_index("model")
results.round(4)

In [ ]:
fm_ids = FM(**{**best["FM"], "use_meta": False}).fit(ts_test)
ablation = pd.DataFrame([
    {"model": "FM, ID + метапризнаки", **evaluate(fitted["FM"], ts_test, gt_test, k=K)},
    {"model": "FM, только ID", **evaluate(fm_ids, ts_test, gt_test, k=K)},
]).set_index("model")
print(ablation.round(4))

known = test[test.user_id.isin(ts_test.user2idx.index) & test.item_id.isin(ts_test.item2idx.index)]
uu = ts_test.user2idx.loc[known.user_id].to_numpy(); ii = ts_test.item2idx.loc[known.item_id].to_numpy()
pred = fitted["timeSVD++"].predict(uu, ii, known.day.to_numpy(dtype=np.float32))
y = known.rating.to_numpy()
print("timeSVD++ RMSE на test:", round(float(np.sqrt(np.mean((np.clip(pred, 1, 5) - y) ** 2))), 4),
      " среднее по train:", round(float(np.sqrt(np.mean((ts_test.r.mean() - y) ** 2))), 4))

## 7. Выводы

Выводы ниже формируются по фактическим числам из таблиц выше.

In [ ]:
r = results.sort_values("NDCG@10", ascending=False)
pop = results.loc["Popular", "NDCG@10"]
out = []
out.append(f"Лучшая модель по NDCG@10: {r.index[0]}, {r.iloc[0]['NDCG@10']:.4f}. Вторая: {r.index[1]}, {r.iloc[1]['NDCG@10']:.4f}.")
br = results["Recall@10"].idxmax()
out.append(f"По Recall@10 лидирует {br}, {results.loc[br, 'Recall@10']:.4f}." + (" Порядок лидеров по двум метрикам совпадает." if br == r.index[0] else " Лидеры по двум метрикам различаются, значит модели по-разному ставят релевантное в начало списка."))
beat = [m for m in results.index if m != "Popular" and results.loc[m, "NDCG@10"] > pop]
out.append(f"Бейзлайн по популярности: NDCG@10 = {pop:.4f}. Его обходят: {', '.join(beat) if beat else 'ни одна модель'}.")
lin = [m for m in ["ItemKNN", "EASE"] if m in results.index]
fac = [m for m in ["iALS", "FM"] if m in results.index]
bl, bf = max(lin, key=lambda m: results.loc[m, "NDCG@10"]), max(fac, key=lambda m: results.loc[m, "NDCG@10"])
out.append(f"Среди item-item моделей лучше {bl}, среди факторизационных {bf}. "
           + ("Item-item подход сильнее: при очень разреженных данных прямые сходства товаров надёжнее латентных факторов." if results.loc[bl, "NDCG@10"] > results.loc[bf, "NDCG@10"]
              else "Факторизация сильнее: латентные факторы лучше обобщают при малой истории пользователей."))
a, b = ablation.iloc[0]["NDCG@10"], ablation.iloc[1]["NDCG@10"]
d = (a - b) / max(b, 1e-12) * 100
out.append(f"Метаданные в FM: NDCG@10 {a:.4f} против {b:.4f} без них, изменение {d:+.1f}%. "
           + ("Бренд, категории и цена помогают: они переносят информацию между редкими товарами и дают профиль пользователю с короткой историей." if d > 0
              else "Метаданные не дали прироста: при наличии ID признаки бренда, категорий и цены слабо добавляют информацию, а профиль категорий частично дублирует ID пользователя."))
rank_t = list(r.index).index("timeSVD++") + 1
rmse_t = float(np.sqrt(np.mean((np.clip(pred, 1, 5) - y) ** 2))); rmse_0 = float(np.sqrt(np.mean((ts_test.r.mean() - y) ** 2)))
out.append(f"timeSVD++ занимает {rank_t} место из {len(r)} по NDCG@10. RMSE на test {rmse_t:.4f} против {rmse_0:.4f} у среднего. "
           "Модель учится предсказывать оценку уже купленного товара, а не факт покупки, поэтому хорошая точность оценок не гарантирует хороший top-10: "
           "наверх попадают товары с высоким средним рейтингом, но малым числом покупок.")
cm, cn = results["Coverage@10"].idxmax(), results["Coverage@10"].idxmin()
out.append(f"Самая разнообразная выдача у {cm}, Coverage@10 = {results.loc[cm, 'Coverage@10']:.3f}, самая узкая у {cn}, {results.loc[cn, 'Coverage@10']:.3f}.")
print("\n\n".join(out))

Ограничения. Выборка 5-core отфильтрована авторами по всей истории, то есть с использованием будущих данных. Метаданные собраны в 2018 году, цена могла отличаться в момент покупки. Холодные пользователи и товары исключены из оценки. Сетки гиперпараметров небольшие, один сид, доверительные интервалы метрик не считались.